### SST-2 sentiment, GPT-2, LoRA

In [ ]:
!pip install -q transformers datasets peft evaluate accelerate scikit-learn

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModelForSequenceClassification
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "gpt2"
NUM_LABELS = 2
PEFT_SAVE_PATH = "./peft_lora_gpt2_saved"

In [ ]:
dataset = load_dataset("glue", "sst2")
train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
eval_dataset = dataset["validation"].shuffle(seed=42).select(range(200))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)
drop_cols = [c for c in ["sentence", "idx"] if c in tokenized_train.column_names]
tokenized_train = tokenized_train.remove_columns(drop_cols)
tokenized_eval = tokenized_eval.remove_columns(drop_cols)
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_eval = tokenized_eval.rename_column("label", "labels")
tokenized_train.set_format("torch")
tokenized_eval.set_format("torch")

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return accuracy_metric.compute(predictions=np.argmax(logits, axis=-1), references=labels)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
foundation_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
foundation_model.config.pad_token_id = tokenizer.pad_token_id
foundation_model = foundation_model.to(device)

baseline_trainer = Trainer(
    model=foundation_model,
    args=TrainingArguments(output_dir="./baseline_eval", per_device_eval_batch_size=16, report_to="none"),
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
baseline_results = baseline_trainer.evaluate()
baseline_results["eval_accuracy"], baseline_results["eval_loss"]

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],
    bias="none",
)

base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
base_model.config.pad_token_id = tokenizer.pad_token_id
peft_model = get_peft_model(base_model, lora_config).to(device)
peft_model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="./peft_lora_gpt2",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    learning_rate=2e-4,
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

peft_trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
peft_trainer.train()

In [ ]:
peft_model.save_pretrained(PEFT_SAVE_PATH)
tokenizer.save_pretrained(PEFT_SAVE_PATH)
os.listdir(PEFT_SAVE_PATH)

In [ ]:
loaded_base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
loaded_base.config.pad_token_id = tokenizer.pad_token_id
loaded_peft_model = PeftModelForSequenceClassification.from_pretrained(loaded_base, PEFT_SAVE_PATH).to(device)
loaded_peft_model.eval()

peft_eval_trainer = Trainer(
    model=loaded_peft_model,
    args=TrainingArguments(output_dir="./peft_eval", per_device_eval_batch_size=16, report_to="none"),
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
peft_results = peft_eval_trainer.evaluate()
peft_results["eval_accuracy"], peft_results["eval_loss"]

In [ ]:
label_map = {0: "NEG", 1: "POS"}

def predict(model, text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = torch.argmax(F.softmax(logits, dim=-1), dim=-1).item()
    return label_map[pred]

examples = [
    "This movie was absolutely wonderful!",
    "I hated every minute of this film.",
    "The acting was ok but the plot was weak.",
]
for s in examples:
    print(s[:50])
    print("  base:", predict(foundation_model, s), "  lora:", predict(loaded_peft_model, s))

In [ ]:
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in foundation_model.parameters())
print("baseline acc", round(baseline_results["eval_accuracy"], 4))
print("lora acc    ", round(peft_results["eval_accuracy"], 4))
print("delta       ", round(peft_results["eval_accuracy"] - baseline_results["eval_accuracy"], 4))
print("trainable params", trainable, "(~{:.2f}% of full model)".format(100 * trainable / total))